# 🫀 CancerNet — Fast Training on Free T4 GPU

**Estimated training time: ~3-4 hours**

### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Have `cancer-detection-code.zip` ready on your Desktop
3. Have `kaggle.json` ready ([kaggle.com/settings](https://www.kaggle.com/settings) → API → Create New Token)

### Run all cells top to bottom. Two cells will ask you to upload files.

---

## Step 1: Verify GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("\n\n⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

!nvidia-smi

## Step 2: Upload project zip 📤

Upload `cancer-detection-code.zip` from your Desktop.

In [ ]:
from google.colab import files
print("Upload your cancer-detection-code.zip:")
uploaded = files.upload()

In [ ]:
import os

# Clean up any previous run
!rm -rf /content/cancer-detection

# Unzip
!unzip -q cancer-detection-code.zip -d /content/cancer-detection

# Handle nested folder if exists
if os.path.exists('/content/cancer-detection/cancer-detection'):
    !cp -r /content/cancer-detection/cancer-detection/* /content/cancer-detection/
    !rm -rf /content/cancer-detection/cancer-detection

# Create directories
!mkdir -p /content/cancer-detection/data/raw
!mkdir -p /content/cancer-detection/data/processed
!mkdir -p /content/cancer-detection/data/splits
!mkdir -p /content/cancer-detection/checkpoints
!mkdir -p /content/cancer-detection/logs
!mkdir -p /content/cancer-detection/results

%cd /content/cancer-detection
print("\n\u2705 Project extracted!")
!ls -la

## Step 3: Install dependencies

In [ ]:
import torch

# Install PyTorch Geometric
!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html

# Install other dependencies
!pip install -q timm albumentations opencv-python scikit-image scikit-learn \
    omegaconf tqdm grad-cam networkx scipy Pillow

print("\n\u2705 All dependencies installed!")

## Step 4: Download dataset from Kaggle 📤

Upload your `kaggle.json` API key below.  
Get it from: [kaggle.com/settings](https://www.kaggle.com/settings) → API → Create New Token

In [ ]:
from google.colab import files
print("Upload your kaggle.json:")
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("\u2705 Kaggle API key configured!")

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d paultimothymooney/breast-histopathology-images -p /content/dataset --unzip
print("\n\u2705 Dataset downloaded!")

In [ ]:
import os, shutil

# Find and copy patient folders to data/raw
dataset_root = '/content/dataset'
found = False

for root, dirs, files_list in os.walk(dataset_root):
    numbered_dirs = [d for d in dirs if d.isdigit()]
    if len(numbered_dirs) > 50:
        print(f"Found {len(numbered_dirs)} patient folders in: {root}")
        for d in numbered_dirs:
            src = os.path.join(root, d)
            dst = os.path.join('/content/cancer-detection/data/raw', d)
            if not os.path.exists(dst):
                shutil.copytree(src, dst)
        found = True
        break

# Also copy IDC_regular folder if exists
for root, dirs, files_list in os.walk(dataset_root):
    for d in dirs:
        if d.startswith('IDC_regular'):
            src = os.path.join(root, d)
            dst = os.path.join('/content/cancer-detection/data/raw', d)
            if not os.path.exists(dst):
                shutil.copytree(src, dst)

total = len(os.listdir('/content/cancer-detection/data/raw'))
print(f"\n\u2705 {total} patient folders ready in data/raw/")

## Step 5: Create fast training config

This config uses **only 20 patients** + larger batch size to finish in ~3-4 hours on T4.

In [ ]:
%%writefile /content/cancer-detection/configs/config_fast.yaml
project:
  name: "CancerNet"
  seed: 42
  device: "cuda"
  mixed_precision: true

data:
  raw_dir: "data/raw"
  processed_dir: "data/processed"
  splits_dir: "data/splits"
  image_size: 224
  num_workers: 2
  pin_memory: true
  max_patients: 20

model:
  efficientnet:
    backbone: "efficientnet_b4"
    pretrained: true
    freeze_epochs: 3
    embedding_dim: 1792
  vit:
    backbone: "vit_base_patch16_224"
    pretrained: true
    freeze_epochs: 3
    embedding_dim: 768
  gnn:
    num_node_features: 7
    hidden_dim: 256
    output_dim: 256
    num_layers: 3
    dropout: 0.3
  fusion:
    input_dim: 2816
    num_heads: 8
    temperature_learnable: true
    dropout: 0.3
  classifier:
    hidden_dims: [512, 128]
    num_classes: 2
    dropout: 0.4

training:
  mixed_precision: true
  epochs: 30
  batch_size: 32
  accumulation_steps: 1
  optimizer:
    name: "adamw"
    lr: 1e-4
    weight_decay: 1e-4
    differential_lr:
      backbone_lr_factor: 0.1
  scheduler:
    name: "cosine_warmup"
    warmup_epochs: 3
    min_lr: 1e-6
  loss:
    name: "focal"
    alpha: 0.75
    gamma: 2.0
    label_smoothing: 0.1
  early_stopping:
    patience: 7
    monitor: "val_auc"

augmentation:
  train:
    stain_normalize: true
    stain_method: "macenko"
  val:
    stain_normalize: true
  tta:
    enabled: true
    n_augments: 10

graph:
  n_neighbors: 8
  node_feature_method: "deep"

logging:
  log_dir: "logs"
  checkpoint_dir: "checkpoints"
  results_dir: "results"
  save_top_k: 3
  log_every_n_steps: 50

## Step 6: Patch code to support max_patients

This patches `dataset.py` and `train.py` to support the `max_patients` config option.

In [ ]:
# ---- Patch dataset.py ----
dataset_path = "/content/cancer-detection/src/utils/dataset.py"
with open(dataset_path, "r") as f:
    code = f.read()

# Add max_patients parameter to get_patient_level_splits
code = code.replace(
    "def get_patient_level_splits(root_dir, val_ratio=0.15, test_ratio=0.15, seed=42):",
    "def get_patient_level_splits(root_dir, val_ratio=0.15, test_ratio=0.15, seed=42, max_patients=None):"
)

# Add subsetting logic after the empty-check
code = code.replace(
    '    train_val, test = train_test_split(\n        all_patients, test_size=test_ratio, random_state=seed)',
    '''    # Limit patients for faster training
    if max_patients is not None and max_patients < len(all_patients):
        import random
        rng = random.Random(seed)
        all_patients = sorted(rng.sample(all_patients, max_patients))
        print(f"SUBSET MODE: using {max_patients} patients")

    train_val, test = train_test_split(
        all_patients, test_size=test_ratio, random_state=seed)'''
)

# Add max_patients parameter to get_dataloaders
code = code.replace(
    "def get_dataloaders(root_dir, train_transform, val_transform,\n                    batch_size=32, num_workers=0, seed=42, image_size=224):",
    "def get_dataloaders(root_dir, train_transform, val_transform,\n                    batch_size=32, num_workers=0, seed=42, image_size=224, max_patients=None):"
)

# Pass max_patients to the splits function
code = code.replace(
    "train_ids, val_ids, test_ids = get_patient_level_splits(root_dir, seed=seed)",
    "train_ids, val_ids, test_ids = get_patient_level_splits(root_dir, seed=seed, max_patients=max_patients)"
)

with open(dataset_path, "w") as f:
    f.write(code)

print("\u2705 dataset.py patched!")

# ---- Patch train.py ----
train_path = "/content/cancer-detection/scripts/train.py"
with open(train_path, "r") as f:
    code = f.read()

# Add max_patients reading from config and passing to get_dataloaders
code = code.replace(
    '    train_loader, val_loader, test_loader = get_dataloaders(\n        root_dir=cfg.data.raw_dir,',
    '    max_patients = cfg.data.get("max_patients", None)\n\n    train_loader, val_loader, test_loader = get_dataloaders(\n        root_dir=cfg.data.raw_dir,'
)

code = code.replace(
    '        image_size=cfg.data.image_size,\n    )',
    '        image_size=cfg.data.image_size,\n        max_patients=max_patients,\n    )'
)

with open(train_path, "w") as f:
    f.write(code)

print("\u2705 train.py patched!")
print("\n\u2705 All code patches applied. Ready to train!")

## Step 7: Sanity check

In [ ]:
%cd /content/cancer-detection

import os
print("Project files:")
for item in sorted(os.listdir('.')):
    if item not in ['venv', '__pycache__', '.git']:
        print(f"  {item}")

print(f"\nPatient folders in data/raw: {len(os.listdir('data/raw'))}")

# Verify config
from omegaconf import OmegaConf
cfg = OmegaConf.load('configs/config_fast.yaml')
print(f"\n--- Config Summary ---")
print(f"  Device        : {cfg.project.device}")
print(f"  Mixed precision: {cfg.training.mixed_precision}")
print(f"  Batch size    : {cfg.training.batch_size}")
print(f"  Max patients  : {cfg.data.max_patients}")
print(f"  Epochs (max)  : {cfg.training.epochs}")
print(f"  Early stopping: patience {cfg.training.early_stopping.patience}")
print(f"\n\u2705 Everything looks good!")

## Step 8: 🚀 Start Training!

**Expected time: ~3-4 hours on T4 GPU.**

Keep this browser tab open so Colab doesn't disconnect!

In [ ]:
%cd /content/cancer-detection
!python scripts/train.py --config configs/config_fast.yaml

## Step 9: Download trained model

After training completes, download your best model checkpoint.

In [ ]:
import os
from google.colab import files

if os.path.exists('checkpoints/best_model.pth'):
    # Show model info
    import torch
    ckpt = torch.load('checkpoints/best_model.pth', map_location='cpu')
    print(f"Best model from epoch {ckpt['epoch']}")
    print(f"Metrics: {ckpt['metrics']}")
    print(f"\nDownloading...")
    files.download('checkpoints/best_model.pth')
else:
    print("⚠️ No best_model.pth found. Listing available checkpoints:")
    !ls -la checkpoints/

In [ ]:
# Download everything (checkpoints + logs + results) as a zip
!zip -r /content/training_results.zip checkpoints/ logs/ results/
files.download('/content/training_results.zip')
print("\u2705 All results downloaded!")

---
## ℹ️ Tips & Troubleshooting

| Issue | Solution |
|-------|----------|
| **CUDA out of memory** | Change `batch_size` from 32 to 16 in Step 5 config |
| **Colab disconnects** | Keep tab open, training auto-saves checkpoints |
| **Want more accuracy** | Change `max_patients` from 20 to 50 or remove it (will take longer) |
| **Want even faster** | Change `max_patients` to 10 (~1-2 hours) |